# Policy RAG Assistant

An AI-powered assistant that answers employee questions about **expense, travel,
finance, and HR policy** using Retrieval-Augmented Generation (RAG). It answers
strictly from the provided documents, always cites its sources, supports
multi-turn follow-up questions, and clearly says so when the answer isn't in
the documents rather than guessing.

**This notebook is fully self-contained** — every module from the original
project (`document_loader`, `chunker`, `bm25`, `indexer`, `retriever`,
`generator`, `rag_pipeline`) is inlined below as its own cell, in the order
data flows through the pipeline. Run all cells top to bottom.

**Assumption:** the assignment named four documents (Expense Policy, Travel
Policy, Finance Policy, Employee Handbook) but didn't attach their content, so
realistic sample versions are embedded below as strings. To use your **real**
documents instead, see the "Loading your own documents" cell near the bottom
— swap in your files, no other code changes needed.

**No API key required.** Without `ANTHROPIC_API_KEY` set, the generator falls
back to an extractive mode that returns the top-matching passages directly
with citations — every other part of the pipeline (retrieval, citation,
refusal-when-ungrounded, conversation/follow-ups, hybrid search, metadata
filtering, query expansion, reranking, incremental indexing) is fully
exercised either way. Set the key beforehand to get fluent LLM-synthesized
answers instead.

```python
import os
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # optional
```

## Architecture

```
documents ──▶ chunker ──▶ index (TF-IDF + BM25) ──▶ hybrid retriever
                                                          │
question ──▶ follow-up query rewrite ────────────────────┘
                                                          ▼
                                          confidence gate → answer + citations
                                          (LLM if configured, else extractive)
```


In [ ]:
!pip install -q scikit-learn numpy anthropic 2>/dev/null || true

## 1. Imports

In [ ]:
from __future__ import annotations

import math
import os
import re
import textwrap
from collections import Counter
from dataclasses import dataclass, field
from typing import Dict, List, Optional

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Sample source documents

Realistic sample policy text standing in for the real documents (see assumption above). Each is a Markdown string with `##`/`###` headers, which the chunker below uses to keep every retrieved passage aligned to one policy section for clean citations.

In [ ]:
EXPENSE_POLICY = """# Expense Policy

## 1. Purpose and Scope
This policy governs how employees incur, document, and get reimbursed for business
expenses. It applies to all full-time and part-time employees, and to contractors
where explicitly stated in their contract.

## 2. General Principles
Employees should spend company money as carefully as they would spend their own.
All expenses must be reasonable, necessary for business purposes, and properly
documented with an itemized receipt.

## 3. Reimbursable Expense Categories
### 3.1 Meals
- Individual meals while traveling on company business are reimbursable up to
  INR 1,500 per day (breakfast, lunch, and dinner combined).
- Client meals require the client's name and business purpose noted on the expense
  report and are reimbursable up to INR 5,000 per meal with manager approval.
- Alcohol is not reimbursable except at client dinners, and even then capped at
  20% of the total client meal bill.

### 3.2 Local Transportation
- Cab, auto, and metro fares for business purposes are reimbursable in full with
  receipts or app-based trip summaries.
- Personal vehicle mileage is reimbursed at INR 12 per kilometer for business travel
  beyond the normal commute.

### 3.3 Office Supplies and Equipment
- Purchases under INR 3,000 can be self-approved and reimbursed.
- Purchases above INR 3,000 require prior written approval from the employee's
  manager before the purchase is made.

### 3.4 Software and Subscriptions
- Any recurring software subscription must be approved by the employee's manager
  and IT before purchase, and reviewed annually for continued need.

## 4. Non-Reimbursable Expenses
The following are never reimbursable: parking or traffic fines, personal
entertainment, gym memberships, clothing (except required safety gear), and
expenses lacking a valid receipt over INR 500.

## 5. Submission Process
- Expense reports must be submitted within 30 days of the expense being incurred.
- Reports are submitted through the Finance Portal with scanned or photographed
  itemized receipts attached.
- Reports older than 60 days will not be reimbursed except with VP-level approval.

## 6. Approval Workflow
- Expenses under INR 10,000 require single manager approval.
- Expenses between INR 10,000 and INR 50,000 require manager plus department head
  approval.
- Expenses above INR 50,000 require Finance Business Partner sign-off in addition
  to department head approval.

## 7. Reimbursement Timeline
Approved expense reports are reimbursed within 10 business days via the same
payroll cycle bank transfer used for salary.

## 8. Policy Violations
Repeated policy violations (e.g., submitting personal expenses as business
expenses) may result in disciplinary action up to and including termination, per
the Employee Handbook's conduct guidelines.
"""

TRAVEL_POLICY = """# Travel Policy

## 1. Purpose
This policy defines rules for domestic and international business travel,
including booking procedures, class of travel, per-diems, and safety
requirements.

## 2. Trip Approval
- All business travel must be approved by the employee's manager at least 5
  business days before domestic trips and 10 business days before international
  trips.
- Emergency travel (e.g., client escalation) can be approved retroactively within
  48 hours by the manager, with justification noted in the travel request.

## 3. Booking Procedure
- All flights, trains, and hotels must be booked through the company's approved
  travel portal (TravelDesk) to ensure duty-of-care tracking.
- Employees who book outside TravelDesk for valid reasons must notify HR-Travel
  within 24 hours for safety tracking purposes.

## 4. Class of Travel
### 4.1 Air Travel
- Domestic flights: Economy class for all employees.
- International flights under 6 hours: Economy class.
- International flights over 6 hours: Premium Economy for employees at Manager
  level and above; Economy for others.
- Business class requires VP approval regardless of flight duration.

### 4.2 Rail Travel
- AC 2-tier or equivalent for journeys under 8 hours; AC 1st class or equivalent
  permitted for journeys over 8 hours with manager approval.

## 5. Accommodation
- Hotel bookings should not exceed INR 8,000 per night in metro cities and
  INR 5,000 per night in non-metro cities, unless the conference venue hotel
  costs more (in which case the venue hotel is allowed).
- Extended stays beyond the business need must be pre-approved and the
  additional nights are the employee's personal expense.

## 6. Per Diem
- Domestic travel: INR 1,500 per day for incidentals (meals, local transport)
  in lieu of itemized receipts, employee's choice between per diem or itemized
  reimbursement under the Expense Policy, not both.
- International travel: per diem varies by country and is published on the
  Finance intranet page, typically USD 60-90 per day.

## 7. Travel Insurance
All company-booked international trips automatically include travel medical
insurance through the corporate policy. Employees should carry the insurance
card, obtainable from HR-Travel before departure.

## 8. Combining Business and Personal Travel
Employees may extend a business trip for personal travel. The company will
cover costs only up to what the business-only itinerary would have cost;
employees must show a cost-comparison quote from TravelDesk to substantiate
this.

## 9. Safety and Emergency Contacts
Employees traveling internationally must register their itinerary with
HR-Travel for the 24/7 travel safety hotline. In the event of a natural
disaster, political unrest, or medical emergency, contact the hotline number
provided at booking confirmation.
"""

FINANCE_POLICY = """# Finance Policy

## 1. Purpose
This policy governs financial controls, purchase approvals, vendor payments,
and budget ownership across the company.

## 2. Budget Ownership
- Each department head owns their department's annual budget and is
  accountable for variance explanations during quarterly business reviews.
- Budget transfers between departments above INR 200,000 require CFO approval.

## 3. Purchase Orders
- Any purchase above INR 25,000 requires a Purchase Order (PO) raised in the
  Finance system before the vendor is engaged.
- POs must include cost center, budget line item, and a business justification
  of at least two sentences.

## 4. Vendor Onboarding
- New vendors must complete KYC (PAN, GST, bank details) and be approved by
  Finance Operations before the first payment is processed.
- Vendor payment terms default to Net 30 unless otherwise negotiated and
  documented in the contract.

## 5. Invoice Processing
- Invoices are processed within 5 business days of receipt if they match an
  approved PO and delivery confirmation (a "three-way match").
- Mismatched invoices are routed back to the requester for resolution and do
  not restart the payment clock until resolved.

## 6. Capital Expenditure (CapEx)
- Any single asset purchase above INR 500,000 is classified as CapEx and
  requires CFO and CEO approval, plus a depreciation schedule submitted by
  Finance.
- CapEx requests must be included in the annual budget cycle unless approved
  as an exception by the CFO.

## 7. Corporate Credit Cards
- Corporate cards are issued to employees at Manager level and above who
  travel frequently or manage vendor relationships.
- Cardholders must reconcile statements monthly in the Finance Portal; unreconciled
  balances beyond 60 days may result in card suspension.

## 8. Financial Reporting Calendar
- Monthly close happens on the 5th business day of the following month.
- Department heads must submit budget variance commentary by the 7th business
  day.

## 9. Fraud and Whistleblowing
Any suspected financial fraud, including expense fraud or vendor kickback
schemes, should be reported through the confidential Ethics Hotline described
in the Employee Handbook. Retaliation against good-faith whistleblowers is
strictly prohibited.

## 10. Audit
Finance conducts quarterly internal audits of expense reports and vendor
payments above INR 100,000. Employees must retain original receipts and
supporting documents for 3 years for audit purposes.
"""

EMPLOYEE_HANDBOOK = """# Employee Handbook

## 1. Welcome
This handbook summarizes company-wide policies on conduct, leave, working
hours, and workplace expectations. It complements, and does not replace, your
offer letter and any local labor-law requirements.

## 2. Working Hours
- Standard working hours are 9:30 AM to 6:30 PM, Monday to Friday, with a
  flexible 1-hour window for start time.
- Employees in client-facing roles may be asked to align hours to client time
  zones, with manager agreement.

## 3. Leave Policy
### 3.1 Annual Leave
- Full-time employees accrue 21 days of paid annual leave per calendar year,
  credited at 1.75 days per month.
- Unused leave up to 10 days can be carried forward to the next calendar year;
  any excess lapses on December 31.

### 3.2 Sick Leave
- Employees receive 10 days of paid sick leave per year. Sick leave beyond 3
  consecutive days requires a doctor's certificate.

### 3.3 Parental Leave
- Primary caregivers receive 26 weeks of paid parental leave.
- Secondary caregivers receive 2 weeks of paid parental leave, usable within 6
  months of the child's arrival.

### 3.4 Bereavement Leave
Employees receive up to 5 paid days for the loss of an immediate family
member.

## 4. Code of Conduct
- Employees are expected to treat colleagues, clients, and vendors with
  respect. Harassment, discrimination, or retaliation of any kind will not be
  tolerated.
- Conflicts of interest (e.g., a personal financial stake in a vendor) must be
  disclosed to HR and Finance before engaging with that vendor.

## 5. Remote Work
- Employees may work remotely up to 2 days per week by default; fully remote
  arrangements require manager and HR Business Partner approval.
- Remote employees must be reachable during core hours (11 AM - 4 PM local
  time) for meetings.

## 6. Performance Reviews
Performance reviews happen twice yearly, in June and December. Ratings feed
into the annual compensation review cycle each April.

## 7. Ethics Hotline
Employees can report ethical concerns, including financial fraud, harassment,
or safety violations, confidentially and anonymously through the Ethics
Hotline (phone and web form, details on the HR intranet page). Retaliation
against a good-faith reporter is grounds for disciplinary action against the
retaliator.

## 8. Disciplinary Process
Policy violations are handled through a progressive discipline process:
verbal warning, written warning, final written warning, and termination,
except in cases of gross misconduct (e.g., fraud, harassment, safety
violations) which may result in immediate termination.

## 9. Offboarding
Departing employees must return all company assets (laptop, access cards,
phone) on or before their last working day, and complete a knowledge-transfer
handover with their manager.
"""

RAW_DOCUMENTS = {
    "expense_policy": ("Expense Policy", EXPENSE_POLICY),
    "travel_policy": ("Travel Policy", TRAVEL_POLICY),
    "finance_policy": ("Finance Policy", FINANCE_POLICY),
    "employee_handbook": ("Employee Handbook", EMPLOYEE_HANDBOOK),
}

## 3. Document loading + chunking

`load_sample_documents()` uses the embedded strings above. `load_documents_from_dir(path)` is provided as an alternative for loading your **real** `.md`/`.txt` policy files from disk instead — see the usage example near the end of the notebook.

Chunking splits on Markdown headers first (so a chunk stays aligned to one policy topic, e.g. "Travel Policy > 6. Per Diem"), and only falls back to overlapping word-windows for sections that are still long.

In [ ]:
@dataclass
class Document:
    doc_id: str
    title: str
    doc_type: str
    text: str = field(repr=False)


@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    doc_title: str
    doc_type: str
    section_title: str
    text: str


def load_sample_documents() -> List[Document]:
    """Loads the built-in sample documents defined above."""
    return [
        Document(doc_id=doc_id, title=title, doc_type=doc_id, text=text)
        for doc_id, (title, text) in RAW_DOCUMENTS.items()
    ]


def load_documents_from_dir(data_dir: str) -> List[Document]:
    """Loads .md/.txt files from a directory instead of the built-in samples.
    Use this to plug in your REAL policy documents -- filenames containing
    'expense', 'travel', 'finance', or 'handbook' are auto-tagged; anything
    else is tagged 'general'."""
    doc_type_keywords = {
        "expense": "expense_policy",
        "travel": "travel_policy",
        "finance": "finance_policy",
        "handbook": "employee_handbook",
    }
    docs = []
    for filename in sorted(os.listdir(data_dir)):
        if not filename.lower().endswith((".md", ".txt")):
            continue
        path = os.path.join(data_dir, filename)
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        lower = filename.lower()
        doc_type = next(
            (v for k, v in doc_type_keywords.items() if k in lower), "general"
        )
        title = os.path.splitext(filename)[0].replace("_", " ").replace("-", " ").title()
        docs.append(
            Document(doc_id=os.path.splitext(filename)[0], title=title, doc_type=doc_type, text=text)
        )
    return docs


HEADER_RE = re.compile(r"^(#{1,6})\s+(.*)$", re.MULTILINE)


def _split_by_headers(text: str):
    matches = list(HEADER_RE.finditer(text))
    if not matches:
        yield "Full Document", text.strip()
        return
    stack: List[str] = []
    for i, m in enumerate(matches):
        level = len(m.group(1))
        title = m.group(2).strip()
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        stack = stack[: level - 1]
        stack.append(title)
        header_path = " > ".join(stack)
        if body:
            yield header_path, body


def _word_windows(words: List[str], max_words: int, overlap: int):
    step = max_words - overlap
    for start in range(0, len(words), step):
        window = words[start : start + max_words]
        if not window:
            continue
        yield " ".join(window)
        if start + max_words >= len(words):
            break


def chunk_document(doc: Document, max_words: int = 180, overlap: int = 30) -> List[Chunk]:
    chunks: List[Chunk] = []
    idx = 0
    for header_path, section_text in _split_by_headers(doc.text):
        words = section_text.split()
        pieces = [section_text] if len(words) <= max_words else list(_word_windows(words, max_words, overlap))
        for piece in pieces:
            idx += 1
            chunks.append(
                Chunk(
                    chunk_id=f"{doc.doc_id}::{idx}",
                    doc_id=doc.doc_id,
                    doc_title=doc.title,
                    doc_type=doc.doc_type,
                    section_title=header_path,
                    text=piece.strip(),
                )
            )
    return chunks


def chunk_documents(docs: List[Document], **kwargs) -> List[Chunk]:
    all_chunks: List[Chunk] = []
    for doc in docs:
        all_chunks.extend(chunk_document(doc, **kwargs))
    return all_chunks

## 4. BM25 (keyword search signal)

A minimal, dependency-free Okapi BM25 implementation — no `rank_bm25` package needed. This is one half of the **Hybrid Search** bonus feature; the other half (TF-IDF cosine similarity) is built into the index below.

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9]+")

STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "of", "in", "on", "at", "to", "for", "with", "and", "or", "but", "if",
    "what", "which", "who", "whom", "this", "that", "these", "those", "it",
    "as", "by", "from", "do", "does", "did", "can", "could", "would",
    "should", "will", "shall", "may", "might", "i", "you", "he", "she",
    "we", "they", "my", "your", "his", "her", "our", "their", "current",
}


def tokenize(text: str) -> List[str]:
    return [t for t in TOKEN_RE.findall(text.lower()) if t not in STOPWORDS]


class BM25:
    def __init__(self, corpus: List[List[str]], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.corpus = corpus
        self.doc_lengths = [len(doc) for doc in corpus]
        self.avg_doc_len = sum(self.doc_lengths) / len(self.doc_lengths) if corpus else 0.0
        self.doc_freqs: List[Counter] = [Counter(doc) for doc in corpus]
        self.n_docs = len(corpus)
        self.idf: Dict[str, float] = self._compute_idf()

    def _compute_idf(self) -> Dict[str, float]:
        df: Counter = Counter()
        for doc in self.corpus:
            for term in set(doc):
                df[term] += 1
        return {
            term: math.log(1 + (self.n_docs - freq + 0.5) / (freq + 0.5))
            for term, freq in df.items()
        }

    def get_scores(self, query_tokens: List[str]) -> List[float]:
        scores = [0.0] * self.n_docs
        for term in query_tokens:
            if term not in self.idf:
                continue
            term_idf = self.idf[term]
            for i, freqs in enumerate(self.doc_freqs):
                f = freqs.get(term, 0)
                if f == 0:
                    continue
                dl = self.doc_lengths[i]
                denom = f + self.k1 * (1 - self.b + self.b * dl / (self.avg_doc_len or 1))
                scores[i] += term_idf * (f * (self.k1 + 1)) / denom
        return scores

## 5. Index

Builds a TF-IDF matrix (scikit-learn) and a BM25 index over the same chunk store. `add_documents()` is the **Incremental Indexing** bonus feature — it appends new chunks and rebuilds statistics without re-reading or re-chunking existing documents.

In [ ]:
class RagIndex:
    def __init__(self):
        self.chunks: List[Chunk] = []
        self.vectorizer: Optional[TfidfVectorizer] = None
        self.tfidf_matrix = None
        self.bm25: Optional[BM25] = None

    def build_from_documents(self, docs: List[Document]) -> None:
        self.chunks = chunk_documents(docs)
        self._rebuild_statistics()

    def add_documents(self, docs: List[Document]) -> int:
        """Incremental indexing: append new docs' chunks and rebuild stats,
        without re-reading/re-chunking existing documents."""
        new_chunks = chunk_documents(docs)
        self.chunks.extend(new_chunks)
        self._rebuild_statistics()
        return len(new_chunks)

    def _rebuild_statistics(self) -> None:
        texts = [c.text for c in self.chunks]
        if not texts:
            self.vectorizer, self.tfidf_matrix, self.bm25 = None, None, None
            return
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_df=0.9)
        self.tfidf_matrix = self.vectorizer.fit_transform(texts)
        self.bm25 = BM25([tokenize(t) for t in texts])

    def stats(self) -> dict:
        by_doc: Dict[str, int] = {}
        for c in self.chunks:
            by_doc[c.doc_title] = by_doc.get(c.doc_title, 0) + 1
        return {"total_chunks": len(self.chunks), "chunks_per_document": by_doc}

## 6. Retriever

Combines TF-IDF and BM25 into one hybrid score (**Hybrid Search**), auto-detects which policy document a query concerns (**Metadata Filtering**), expands the query with a small synonym table (**Query Expansion**), and applies a lightweight lexical rerank pass (**Reranking**) — all without needing a downloaded embedding model, so this runs fully offline.

Note on scoring: TF-IDF cosine and BM25 are combined on a fixed, absolute scale (not normalized against each query's own max score). Per-query-max normalization would make the single "least irrelevant" chunk in the whole corpus score close to 1.0 for every query — including a completely off-topic one — which would break the confidence gate used later to detect "not found in the documents".

In [ ]:
SYNONYMS: Dict[str, List[str]] = {
    "reimburse": ["refund", "repay", "reimbursement"],
    "reimbursement": ["refund", "repay", "reimburse"],
    "flight": ["air travel", "airfare", "plane"],
    "hotel": ["accommodation", "lodging", "stay"],
    "boss": ["manager", "supervisor"],
    "manager": ["supervisor", "boss"],
    "vacation": ["annual leave", "pto", "time off"],
    "sick": ["illness", "medical leave"],
    "cab": ["taxi", "auto", "ride"],
    "laptop": ["equipment", "asset", "device"],
    "fraud": ["misconduct", "embezzlement"],
    "per diem": ["daily allowance", "incidentals"],
}

DOC_TYPE_HINTS: Dict[str, List[str]] = {
    "expense_policy": ["expense", "reimburs", "receipt", "meal", "mileage"],
    "travel_policy": ["travel", "flight", "hotel", "trip", "visa", "booking"],
    "finance_policy": ["invoice", "vendor", "purchase order", "po ", "capex", "budget"],
    "employee_handbook": ["leave", "handbook", "conduct", "working hours", "hr "],
}

BM25_SCALE = 4.0  # fixed scaling divisor -- see note in HybridRetriever.search


def expand_query(query: str) -> str:
    lower = query.lower()
    extra_terms: List[str] = []
    for term, synonyms in SYNONYMS.items():
        if term in lower:
            extra_terms.extend(synonyms)
    return query + " " + " ".join(extra_terms) if extra_terms else query


def infer_doc_type_filter(query: str) -> Optional[str]:
    lower = query.lower()
    hits = {dt: sum(1 for kw in kws if kw in lower) for dt, kws in DOC_TYPE_HINTS.items()}
    hits = {k: v for k, v in hits.items() if v > 0}
    return next(iter(hits)) if len(hits) == 1 else None


def _scale_bm25(scores: np.ndarray) -> np.ndarray:
    if scores.size == 0:
        return scores
    return np.clip(scores / BM25_SCALE, 0.0, 1.0)


@dataclass
class RetrievedChunk:
    chunk: Chunk
    score: float
    overlap_ratio: float = 0.0


class HybridRetriever:
    def __init__(self, index: RagIndex, tfidf_weight: float = 0.5):
        self.index = index
        self.tfidf_weight = tfidf_weight

    def search(
        self,
        query: str,
        top_k: int = 5,
        doc_type_filter: Optional[str] = None,
        use_query_expansion: bool = True,
        auto_filter: bool = True,
    ) -> List[RetrievedChunk]:
        if self.index.vectorizer is None or not self.index.chunks:
            return []

        if auto_filter and doc_type_filter is None:
            doc_type_filter = infer_doc_type_filter(query)

        search_query = expand_query(query) if use_query_expansion else query

        q_vec = self.index.vectorizer.transform([search_query])
        tfidf_scores = cosine_similarity(q_vec, self.index.tfidf_matrix).flatten()
        bm25_scores = np.array(self.index.bm25.get_scores(tokenize(search_query)))

        # Note: tfidf_scores (cosine) is already on an absolute [0, 1] scale,
        # and bm25 is rescaled with a FIXED divisor (not this query's own max).
        # Normalizing by the query's own max would make the single "least
        # irrelevant" chunk in the corpus score ~1.0 for every query, even a
        # totally off-topic one -- which would break the confidence gate
        # used later to say "insufficient information".
        bm25_scaled = _scale_bm25(bm25_scores)
        combined = self.tfidf_weight * tfidf_scores + (1 - self.tfidf_weight) * bm25_scaled

        candidates = list(enumerate(combined))
        if doc_type_filter:
            candidates = [(i, s) for i, s in candidates if self.index.chunks[i].doc_type == doc_type_filter]
        candidates.sort(key=lambda x: x[1], reverse=True)
        top_candidates = candidates[: max(top_k * 3, top_k)]

        reranked = self._rerank(query, top_candidates)
        q_terms = set(tokenize(query))
        results = []
        for i, score in reranked[:top_k]:
            chunk = self.index.chunks[i]
            chunk_terms = set(tokenize(chunk.text))
            ratio = len(q_terms & chunk_terms) / (len(q_terms) or 1)
            results.append(RetrievedChunk(chunk=chunk, score=score, overlap_ratio=ratio))
        return results

    def _rerank(self, query: str, candidates: List[tuple]) -> List[tuple]:
        """Lightweight lexical rerank: a multiplicative boost (not additive)
        so a near-zero base-relevance chunk can't be pushed above the
        confidence threshold purely by incidental word overlap."""
        q_terms = set(tokenize(query))
        boosted = []
        for i, score in candidates:
            chunk_terms = set(tokenize(self.index.chunks[i].text))
            overlap = len(q_terms & chunk_terms) / (len(q_terms) or 1)
            phrase_bonus = 0.15 if query.lower() in self.index.chunks[i].text.lower() else 0.0
            boosted.append((i, score * (1 + 0.3 * overlap) + score * phrase_bonus))
        boosted.sort(key=lambda x: x[1], reverse=True)
        return boosted

## 7. Generator

Turns retrieved chunks into a cited answer. Key pieces:
- **Confidence gating**: a candidate answer is only produced if the top chunk clears both an absolute hybrid-score threshold *and* a minimum lexical-overlap ratio with the query — otherwise the assistant says it couldn't find sufficient information, instead of guessing.
- **Follow-up support**: an elliptical follow-up ("What about international trips?") is rewritten into a standalone query using conversation history, either via an LLM call (if configured) or a heuristic that only fires on genuine continuation cues — not on every short question.
- **Pluggable LLM backend**: uses the Anthropic API if `ANTHROPIC_API_KEY` is set, else falls back to an extractive mode that surfaces the retrieved passages directly with citations.

In [ ]:
CONFIDENCE_THRESHOLD = 0.12
MIN_OVERLAP_RATIO = 0.3


@dataclass
class Turn:
    role: str
    content: str


@dataclass
class ConversationState:
    history: List[Turn] = field(default_factory=list)

    def add(self, role: str, content: str) -> None:
        self.history.append(Turn(role, content))

    def recent_text(self, n_turns: int = 4) -> str:
        return "\n".join(f"{t.role}: {t.content}" for t in self.history[-n_turns:])


def _tokenize_simple(text: str) -> List[str]:
    return re.findall(r"[a-z']+", text.lower())


def _get_llm_client():
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        return None
    try:
        import anthropic  # type: ignore
    except ImportError:
        return None
    try:
        return anthropic.Anthropic(api_key=api_key)
    except Exception:
        return None


def _llm_rewrite_query(client, question: str, conversation: ConversationState) -> str:
    prompt = (
        "Rewrite the follow-up question as a standalone question, using the "
        "conversation history for context. Reply with ONLY the rewritten "
        f"question, nothing else.\n\nConversation so far:\n{conversation.recent_text()}\n\n"
        f"Follow-up question: {question}"
    )
    resp = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=100, messages=[{"role": "user", "content": prompt}]
    )
    text = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
    return text.strip() or question


def rewrite_followup_query(question: str, conversation: ConversationState) -> str:
    if not conversation.history:
        return question

    client = _get_llm_client()
    if client is not None:
        try:
            return _llm_rewrite_query(client, question, conversation)
        except Exception:
            pass

    last_user = next((t.content for t in reversed(conversation.history) if t.role == "user"), "")
    lower = question.lower().strip()
    continuation_starts = ("what about", "how about", "and ", "what if", "and what about", "also ", "what happens if")
    reference_words = {"it", "that", "this", "those", "them", "there"}
    looks_like_followup = lower.startswith(continuation_starts) or (
        len(question.split()) <= 5 and reference_words & set(_tokenize_simple(lower))
    )
    if looks_like_followup and last_user:
        return f"{last_user} {question}"
    return question


def _format_context(chunks: List[RetrievedChunk]) -> str:
    return "\n\n".join(
        f"[{i}] Source: {rc.chunk.doc_title} > {rc.chunk.section_title}\n{rc.chunk.text}"
        for i, rc in enumerate(chunks, start=1)
    )


def _citation_label(rc: RetrievedChunk) -> str:
    return f"{rc.chunk.doc_title} \u00a7 {rc.chunk.section_title}"


@dataclass
class AnswerResult:
    answer: str
    citations: List[str]
    grounded: bool


def _extractive_generate(question: str, retrieved: List[RetrievedChunk]) -> AnswerResult:
    lines = [
        "Here is the most relevant information found in the documents "
        "(no LLM key configured, so passages are shown directly rather than "
        "synthesized into prose):",
        "",
    ]
    citations = []
    for i, rc in enumerate(retrieved[:3], start=1):
        label = _citation_label(rc)
        citations.append(label)
        lines.append(f"{i}. ({label}) {rc.chunk.text}")
        lines.append("")
    return AnswerResult(answer="\n".join(lines).strip(), citations=citations, grounded=True)


def _llm_generate(client, question: str, retrieved: List[RetrievedChunk], conversation) -> AnswerResult:
    context = _format_context(retrieved)
    history_text = conversation.recent_text() if conversation else ""
    system_prompt = textwrap.dedent(
        """
        You are a company policy assistant. Answer ONLY using the numbered
        context passages provided. Every factual claim must be followed by
        its passage number in brackets, e.g. [1]. If the passages do not
        contain the answer, say so plainly instead of guessing. Be concise.
        """
    ).strip()
    user_prompt = f"Conversation so far:\n{history_text}\n\nContext passages:\n{context}\n\nQuestion: {question}"
    resp = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=600, system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    )
    text = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
    used_indices = {int(n) for n in re.findall(r"\[(\d+)\]", text)}
    citations = [_citation_label(rc) for i, rc in enumerate(retrieved, start=1) if i in used_indices] or [
        _citation_label(rc) for rc in retrieved[:2]
    ]
    return AnswerResult(answer=text.strip(), citations=citations, grounded=True)


def generate_answer(
    question: str, retrieved: List[RetrievedChunk], conversation: Optional[ConversationState] = None
) -> AnswerResult:
    if not retrieved or retrieved[0].score < CONFIDENCE_THRESHOLD or retrieved[0].overlap_ratio < MIN_OVERLAP_RATIO:
        return AnswerResult(
            answer=(
                "I could not find sufficient information in the provided documents "
                "(Expense Policy, Travel Policy, Finance Policy, Employee Handbook) to "
                "answer this question confidently. Please rephrase, or check with "
                "HR/Finance directly."
            ),
            citations=[],
            grounded=False,
        )

    client = _get_llm_client()
    if client is not None:
        try:
            return _llm_generate(client, question, retrieved, conversation)
        except Exception as exc:
            fallback = _extractive_generate(question, retrieved)
            fallback.answer += f"\n\n[Note: LLM generation unavailable ({exc}); showing extractive fallback answer instead.]"
            return fallback

    return _extractive_generate(question, retrieved)

## 8. Pipeline

Wires everything above together behind a simple `.ask()` API with built-in conversation state — this is the class you actually interact with.

In [ ]:
@dataclass
class ChatResult:
    question: str
    standalone_question: str
    answer: str
    citations: List[str]
    grounded: bool
    retrieved: List[RetrievedChunk] = field(default_factory=list)


class RagPipeline:
    def __init__(self, documents: Optional[List[Document]] = None):
        self.index = RagIndex()
        self.index.build_from_documents(documents or load_sample_documents())
        self.retriever = HybridRetriever(self.index)
        self.conversation = ConversationState()

    def add_document(self, doc: Document) -> int:
        return self.index.add_documents([doc])

    def reset_conversation(self) -> None:
        self.conversation = ConversationState()

    def ask(
        self, question: str, top_k: int = 5, doc_type_filter: Optional[str] = None, use_conversation: bool = True
    ) -> ChatResult:
        conv = self.conversation if use_conversation else ConversationState()
        standalone_q = rewrite_followup_query(question, conv)
        retrieved = self.retriever.search(standalone_q, top_k=top_k, doc_type_filter=doc_type_filter)
        result = generate_answer(standalone_q, retrieved, conv)
        conv.add("user", question)
        conv.add("assistant", result.answer)
        return ChatResult(
            question=question,
            standalone_question=standalone_q,
            answer=result.answer,
            citations=result.citations,
            grounded=result.grounded,
            retrieved=retrieved,
        )

## 9. Build the index and run the sample questions

Includes a deliberately out-of-scope question (stock price) to demonstrate the refusal behavior, and a follow-up question ("What about international trips?") to demonstrate conversation support.

In [ ]:
pipeline = RagPipeline()
print(pipeline.index.stats())

SAMPLE_QUESTIONS = [
    "What is the daily meal reimbursement limit for domestic travel?",
    "What about international trips?",
    "How many days of annual leave do full-time employees get?",
    "What is the approval process for a purchase order above 25000 rupees?",
    "Can I expense my gym membership?",
    "What is the company's stock price target for next year?",
]

for q in SAMPLE_QUESTIONS:
    result = pipeline.ask(q)
    print(f"Q: {q}")
    if result.standalone_question != q:
        print(f"   (rewritten as: '{result.standalone_question}')")
    print(f"A: {result.answer}")
    if result.citations:
        print("Citations:", "; ".join(result.citations))
    print("=" * 80)

## 10. Interactive chat (optional)

Run this cell to ask your own questions in a loop. Type `reset` to clear
conversation history, or `exit` to stop.


In [ ]:
while True:
    question = input("You: ").strip()
    if not question:
        continue
    if question.lower() in {"exit", "quit"}:
        break
    if question.lower() == "reset":
        pipeline.reset_conversation()
        print("(conversation history cleared)\n")
        continue
    result = pipeline.ask(question)
    print(f"\nAssistant: {result.answer}")
    if result.citations:
        print("Citations:", "; ".join(result.citations))
    print()


## 11. Loading your own documents

To use your **real** policy documents instead of the built-in samples:

1. Upload your `.md`/`.txt` files to a folder (e.g. `./data` in this
   notebook's working directory). Filenames containing "expense", "travel",
   "finance", or "handbook" are auto-tagged for metadata filtering;
   everything else is tagged `general`.
2. Run the cell below.

No other code changes are needed — the same chunker, index, retriever, and
generator all work identically on your real content.


In [ ]:
# import os
# real_docs = load_documents_from_dir("./data")
# pipeline = RagPipeline(documents=real_docs)
# print(pipeline.index.stats())


## 12. Incremental indexing demo (bonus feature)

Adds one new document to the already-built index without re-chunking the
four existing documents, then confirms the new content is immediately
searchable.


In [ ]:
new_doc = Document(
    doc_id="security_policy",
    title="Security Policy",
    doc_type="general",
    text="# Security Policy\n\n## Passwords\nPasswords must be rotated every 90 days.\n",
)
before = pipeline.index.stats()["total_chunks"]
n_added = pipeline.add_document(new_doc)
after = pipeline.index.stats()["total_chunks"]
print(f"Chunks before: {before}, added: {n_added}, after: {after}")

result = pipeline.ask("How often must passwords be rotated?", use_conversation=False)
print(f"\nA: {result.answer}")
print("Citations:", "; ".join(result.citations))
